# Calibration & Uncertainty

Companion notebook for the [Calibration & Uncertainty lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/07-calibration-and-uncertainty).

We build a deliberately **overconfident** classifier, measure its **Expected Calibration Error**
with a reliability diagram, fix it with **temperature scaling** (confirming accuracy is unchanged),
and separate **epistemic** uncertainty with a small ensemble. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)
sigmoid = lambda x: 1 / (1 + np.exp(-x))

## 1 — An overconfident classifier

Margins z give the *true* accuracy sigmoid(z), but the model reports sigmoid(z·sharp) — too sharp,
so its confidence outruns its accuracy.

In [ ]:
N, sharp = 2000, 2.3
z = np.abs(rng.normal(0, 1.3, N))                 # predicted-class margin
true_acc = sigmoid(z)
correct = (rng.random(N) < true_acc).astype(int)  # whether the prediction is right
conf = sigmoid(z * sharp)                          # the model's (overconfident) reported confidence
print(f'average confidence: {conf.mean():.3f}')
print(f'average accuracy:   {correct.mean():.3f}  (much lower -> overconfident)')

## 2 — Reliability diagram and ECE

Bin by confidence; per bin compare accuracy to confidence. ECE is the average gap, weighted by bin
population. Bars below the diagonal = overconfidence.

In [ ]:
def ece(conf, correct, bins=10):
    edges = np.linspace(0.5, 1.0, bins + 1)
    total = 0.0
    centers, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1.0 else conf <= hi)
        if m.sum() == 0:
            centers.append((lo+hi)/2); accs.append(np.nan); continue
        acc, c = correct[m].mean(), conf[m].mean()
        total += (m.sum()/len(conf)) * abs(acc - c)
        centers.append(c); accs.append(acc)
    return total, np.array(centers), np.array(accs)

e0, cen, acc = ece(conf, correct)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0.5,1],[0.5,1],'--', color='#2dd4bf', label='perfect calibration')
ax.bar(cen, acc, width=0.04, color='#6366f1', alpha=0.8, label='accuracy per bin')
ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_xlim(0.5,1); ax.set_ylim(0.5,1)
ax.set_title(f'Reliability diagram (ECE = {e0:.3f})'); ax.legend(facecolor='#1a1d27', edgecolor='#444')
plt.tight_layout(); plt.show()

## 3 — Temperature scaling fixes it (without touching accuracy)

Divide the logits by T and re-softmax. We sweep T, find the one minimizing ECE, and confirm the
predicted class (argmax) — hence accuracy — never changes.

In [ ]:
logit = z * sharp                                 # the raw (over-sharp) logit
Ts = np.linspace(0.5, 4, 40)
eces = [ece(sigmoid(logit / T), correct)[0] for T in Ts]
bestT = Ts[int(np.argmin(eces))]
print(f'best temperature: T = {bestT:.2f}  (≈ the true sharpness {sharp})')
print(f'ECE at T=1.0:    {ece(sigmoid(logit), correct)[0]:.3f}')
print(f'ECE at T={bestT:.2f}: {ece(sigmoid(logit/bestT), correct)[0]:.3f}  (much lower)')
# accuracy is unchanged because dividing all logits by T cannot change the argmax
print('accuracy unchanged by temperature scaling:', correct.mean())

## ✏️ Your turn

**Exercise.** Implement `expected_calibration_error(conf, correct, bins)` (the weighted mean
confidence-accuracy gap) and `apply_temperature(logits, T)` (softmax over the binary case, i.e.
`sigmoid(logits / T)`). These are the measure-then-fix pair for calibration.

In [ ]:
def expected_calibration_error(conf, correct, bins=10):
    # TODO(you): bin by confidence in [0.5,1], sum (n_b/N)*|acc_b - conf_b| over non-empty bins
    return ...

def apply_temperature(logits, T):
    # TODO(you): confidence after dividing logits by temperature T (binary: sigmoid)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(expected_calibration_error(conf, correct), e0)        # matches reference
# scaling up the temperature lowers an overconfident model's ECE
assert expected_calibration_error(apply_temperature(logit, bestT), correct) < \
       expected_calibration_error(apply_temperature(logit, 1.0), correct)
# argmax (predicted class) is invariant to temperature -> accuracy preserved
assert np.array_equal(apply_temperature(logit, 1.0) > 0.5, apply_temperature(logit, 3.0) > 0.5)
print('\u2713 ECE and temperature scaling are correct')

<details>
<summary>Solution</summary>

```python
def expected_calibration_error(conf, correct, bins=10):
    edges = np.linspace(0.5, 1.0, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & ((conf < hi) | (hi == 1.0) & (conf <= hi))
        if m.sum():
            total += (m.sum()/len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return total

def apply_temperature(logits, T):
    return 1 / (1 + np.exp(-logits / T))
```

Temperature scaling is the cheapest win in ML calibration: one parameter fit on validation data,
accuracy untouched (the argmax can't move), confidence made honest.

</details>